# 06 - Sentetik Finans Haberleri Testi

Bu notebook, fine-tune edilen modelleri sentetik finans haberleri üzerinde değerlendirmek için hazırlanmıştır.

Test edilecek modeller:

- `original_finbert`
  - Hugging Face üzerinden doğrudan indirilir: `ProsusAI/finbert`
  - Fine-tune edilmiş model değildir.
- `roberta_base`
  - `checkpoints/financial_sentiment_multi_model/roberta_base/final_model`
- `bert_base_uncased`
  - `checkpoints/financial_sentiment_multi_model/bert_base_uncased/final_model`
- `distilbert_base_uncased`
  - `checkpoints/financial_sentiment_multi_model/distilbert_base_uncased/final_model`

Bu notebook hiçbir sonuç dosyası kaydetmez. Sadece ekrana basar:

- Metrics
- Classification Report
- Confusion Matrix
- Normalized Confusion Matrix
- Correct / Wrong
- Accuracy by true label
- High-confidence wrong examples
- Final summary


In [1]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
# ============================================================
# TRAIN EDİLMİŞ MODELLERİ SENTETİK VERİ ÜZERİNDE TEST ET
# KAYIT YOK - SADECE EKRANA BASAR
#
# Modeller:
# - original_finbert: ProsusAI/finbert doğrudan indirilir
# - roberta_base: fine-tuned final_model klasöründen yüklenir
# - bert_base_uncased: fine-tuned final_model klasöründen yüklenir
# - distilbert_base_uncased: fine-tuned final_model klasöründen yüklenir
# ============================================================

from pathlib import Path
import re
import gc
import warnings

import numpy as np
import pandas as pd

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, logging as hf_logging

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 180)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


Torch version: 2.11.0+cpu
CUDA available: False


In [2]:
# ------------------------------------------------------------
# 0) Ayarlar
# ------------------------------------------------------------

PROJECT_DIR = PROJECT_ROOT

# Sentetik Excel dosyalarının bulunduğu klasör
SYNTHETIC_DATA_DIR = paths.SYNTHETIC_NEWS_DIR

# Fine-tuned modellerin bulunduğu checkpoint kökü
CHECKPOINT_ROOT = PROJECT_DIR / "checkpoints" / "financial_sentiment_multi_model"

VALID_LABELS = ["negative", "neutral", "positive"]

LABEL2ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

ID2LABEL = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

# Fine-tuned modeller: mevcut checkpoint path yapısı korunur
MODEL_RUNS = [
    {
        "run_name": "roberta_base",
        "model_dir": CHECKPOINT_ROOT / "roberta_base" / "final_model",
    },
    {
        "run_name": "bert_base_uncased",
        "model_dir": CHECKPOINT_ROOT / "bert_base_uncased" / "final_model",
    },
    {
        "run_name": "distilbert_base_uncased",
        "model_dir": CHECKPOINT_ROOT / "distilbert_base_uncased" / "final_model",
    },
]

# Original FinBERT doğrudan Hugging Face üzerinden indirilecek
FINBERT_MODEL_ID = "ProsusAI/finbert"

# Sentetik Excel dosyaları
FILE_PATTERNS = [
    "positive_batch_*.xlsx",
    "negative_batch_*.xlsx",
    "neutral_batch_*.xlsx",
]

TEXT_COL_PRIORITY = ["text_en", "text"]

MAX_LENGTH = 128
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("PROJECT_DIR:", PROJECT_DIR)
print("SYNTHETIC_DATA_DIR:", SYNTHETIC_DATA_DIR)
print("SYNTHETIC_DATA_DIR exists:", SYNTHETIC_DATA_DIR.exists())
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("CHECKPOINT_ROOT exists:", CHECKPOINT_ROOT.exists())

if not SYNTHETIC_DATA_DIR.exists():
    raise FileNotFoundError(f"Sentetik veri klasörü bulunamadı: {SYNTHETIC_DATA_DIR}")


Device: cpu
PROJECT_DIR: D:\serkan.kaymak\financial_sentiment_thesis
SYNTHETIC_DATA_DIR: D:\serkan.kaymak\financial_sentiment_thesis\db\evaluation\synthetic_financial_news
SYNTHETIC_DATA_DIR exists: True
CHECKPOINT_ROOT: D:\serkan.kaymak\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model
CHECKPOINT_ROOT exists: True


In [3]:
# ------------------------------------------------------------
# 1) Sentetik Excel dosyalarını oku ve annotation_all_df oluştur
# ------------------------------------------------------------

def natural_sort_key(path):
    name = Path(path).name
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", name)]


def normalize_colname(c):
    c = str(c).strip()
    c = re.sub(r"\s+", "_", c)
    return c


def read_synthetic_excel(path):
    path = Path(path)

    xls = pd.ExcelFile(path)
    sheet_name = "Data" if "Data" in xls.sheet_names else xls.sheet_names[0]

    df = pd.read_excel(path, sheet_name=sheet_name)
    df.columns = [normalize_colname(c) for c in df.columns]
    df = df.dropna(how="all").copy()

    drop_cols = []
    for c in df.columns:
        lc = str(c).lower().strip()

        if lc.startswith("unnamed"):
            drop_cols.append(c)
        elif lc == "index":
            drop_cols.append(c)
        elif c in ["Özet", "Değer", "Ozet", "Deger", "Field", "Value"]:
            drop_cols.append(c)

    df = df.drop(columns=drop_cols, errors="ignore")

    df["source_file"] = path.name
    df["source_path"] = str(path)

    return df


all_files = []

for pattern in FILE_PATTERNS:
    matched = sorted(
        [p for p in SYNTHETIC_DATA_DIR.glob(pattern) if not p.name.startswith("~$")],
        key=natural_sort_key
    )

    print(f"{pattern}: {len(matched)} dosya")

    for p in matched[:PREVIEW_ROWS]:
        print("  ", p.name)

    if len(matched) > 5:
        print("   ...")

    all_files.extend(matched)

all_files = sorted(list(set(all_files)), key=natural_sort_key)

print("\nToplam bulunan Excel dosyası:", len(all_files))

if not all_files:
    raise FileNotFoundError(f"Hiç sentetik Excel dosyası bulunamadı: {SYNTHETIC_DATA_DIR}")

frames = []
bad_files = []

required_cols = ["annotation_id", "sample_id", "date", "label"]

for path in all_files:
    try:
        one = read_synthetic_excel(path)

        missing = [c for c in required_cols if c not in one.columns]
        has_text_col = any(c in one.columns for c in TEXT_COL_PRIORITY)

        if missing or not has_text_col:
            bad_files.append({
                "file_name": path.name,
                "missing_required_cols": missing,
                "has_text_col": has_text_col,
                "columns": list(one.columns)
            })
            continue

        frames.append(one)

    except Exception as e:
        bad_files.append({
            "file_name": path.name,
            "problem": repr(e),
            "columns": None
        })

if bad_files:
    print("Problemli dosyalar:")
    display(pd.DataFrame(bad_files))
    raise ValueError("Bazı dosyalar okunamadı veya gerekli kolonlar eksik.")

annotation_all_df = pd.concat(frames, ignore_index=True)

print("\nannotation_all_df shape:", annotation_all_df.shape)

print("\nKolonlar:")
for i, c in enumerate(annotation_all_df.columns):
    print(f"{i:02d} | {c}")

display(annotation_all_df.head(PREVIEW_ROWS))


positive_batch_*.xlsx: 100 dosya
   positive_batch_000.xlsx
   positive_batch_001.xlsx
   positive_batch_002.xlsx
   ...
negative_batch_*.xlsx: 100 dosya
   negative_batch_000.xlsx
   negative_batch_001.xlsx
   negative_batch_002.xlsx
   ...
neutral_batch_*.xlsx: 100 dosya
   neutral_batch_001.xlsx
   neutral_batch_002.xlsx
   neutral_batch_003.xlsx
   ...

Toplam bulunan Excel dosyası: 300

annotation_all_df shape: (3000, 17)

Kolonlar:
00 | annotation_id
01 | sample_id
02 | date
03 | text_en
04 | text_tr
05 | label
06 | label_id
07 | label_confidence
08 | reason_tr
09 | sector
10 | topic
11 | split
12 | source_type
13 | created_at
14 | source_file
15 | source_path
16 | synthetic_note


,annotation_id,sample_id,date,text_en,text_tr,label,label_id,label_confidence,reason_tr,sector,topic,split,source_type,created_at,source_file,source_path,synthetic_note
0,SYN_NEG_ANN_0821,SYN_NEG_HEAD_000821,2026-04-01 00:00:00,A cybersecurity vendor reported slower billings growth as large enterprise deals took longer to close.,"Bir siber güvenlik sağlayıcısı, büyük kurumsal anlaşmaların kapanmasının uzamasıyla faturalama büyümesinin yavaşladığını bildirdi.",negative,0,high,Faturalama büyümesindeki yavaşlama gelir momentumunu olumsuz etkiler.,Information Technology,billings_slowdown,train,synthetic,2026-05-25 17:00:00,negative_batch_000.xlsx,D:\serkan.kaymak\financial_sentiment_thesis\db\evaluation\synthetic_financial_news\negative_batch_000.xlsx,NaN
1,SYN_NEG_ANN_0822,SYN_NEG_HEAD_000822,2026-04-02 00:00:00,A consumer finance company increased reserves after delinquency rates rose in its credit card portfolio.,"Bir tüketici finansmanı şirketi, kredi kartı portföyünde gecikme oranlarının yükselmesi sonrası rezervlerini artırdı.",negative,0,high,Gecikme oranı ve rezerv artışı varlık kalitesi açısından negatiftir.,Financials,delinquency_increase,train,synthetic,2026-05-25 17:00:00,negative_batch_000.xlsx,D:\serkan.kaymak\financial_sentiment_thesis\db\evaluation\synthetic_financial_news\negative_batch_000.xlsx,NaN
2,SYN_NEG_ANN_0823,SYN_NEG_HEAD_000823,2026-04-03 00:00:00,A hospital operator cut earnings guidance after labor expenses remained above management's expectations.,"Bir hastane işletmecisi, işçilik giderlerinin yönetim beklentilerinin üzerinde kalması sonrası kâr beklentisini düşürdü.",negative,0,high,Yüksek işçilik giderleri ve beklenti indirimi kârlılık için negatiftir.,Health Care,guidance_cut,train,synthetic,2026-05-25 17:00:00,negative_batch_000.xlsx,D:\serkan.kaymak\financial_sentiment_thesis\db\evaluation\synthetic_financial_news\negative_batch_000.xlsx,NaN


In [4]:
# ------------------------------------------------------------
# 2) Eval dataframe hazırla
# ------------------------------------------------------------

df_eval = annotation_all_df.copy()

# Text kolonu
if "text_en" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text_en"]
elif "text" in df_eval.columns:
    df_eval["eval_text"] = df_eval["text"]
else:
    raise ValueError("annotation_all_df içinde text_en veya text kolonu bulunamadı.")

df_eval["eval_text"] = (
    df_eval["eval_text"]
    .astype("string")
    .str.strip()
)

# Sentetik sette gold label direkt label kolonundan gelir
if "label" not in df_eval.columns:
    raise ValueError("Sentetik veri içinde label kolonu bulunamadı.")

df_eval["gold_label"] = (
    df_eval["label"]
    .astype("string")
    .str.lower()
    .str.strip()
    .replace(["", "nan", "none", "<na>"], pd.NA)
)

df_eval = df_eval[
    df_eval["eval_text"].notna() &
    (df_eval["eval_text"] != "") &
    df_eval["gold_label"].isin(VALID_LABELS)
].copy()

df_eval = df_eval.reset_index(drop=True)
df_eval["gold_id"] = df_eval["gold_label"].map(LABEL2ID).astype(int)

print("\nEval shape:", df_eval.shape)

print("\nGold label distribution:")
print(df_eval["gold_label"].value_counts().reindex(VALID_LABELS))

print("\nGold label ratio:")
print(df_eval["gold_label"].value_counts(normalize=True).reindex(VALID_LABELS).mul(100).round(2))

show_cols = ["annotation_id", "sample_id", "date", "eval_text", "gold_label", "source_file"]
show_cols = [c for c in show_cols if c in df_eval.columns]

display(df_eval[show_cols].head(PREVIEW_ROWS))



Eval shape: (3000, 20)

Gold label distribution:
gold_label
negative    1000
neutral     1000
positive    1000
Name: count, dtype: int64[pyarrow]

Gold label ratio:
gold_label
negative    33.33
neutral     33.33
positive    33.33
Name: proportion, dtype: double[pyarrow]


,annotation_id,sample_id,date,eval_text,gold_label,source_file
0,SYN_NEG_ANN_0821,SYN_NEG_HEAD_000821,2026-04-01 00:00:00,A cybersecurity vendor reported slower billings growth as large enterprise deals took longer to close.,negative,negative_batch_000.xlsx
1,SYN_NEG_ANN_0822,SYN_NEG_HEAD_000822,2026-04-02 00:00:00,A consumer finance company increased reserves after delinquency rates rose in its credit card portfolio.,negative,negative_batch_000.xlsx
2,SYN_NEG_ANN_0823,SYN_NEG_HEAD_000823,2026-04-03 00:00:00,A hospital operator cut earnings guidance after labor expenses remained above management's expectations.,negative,negative_batch_000.xlsx


In [5]:
# ------------------------------------------------------------
# 3) Metrik fonksiyonları
# ------------------------------------------------------------

def compute_full_metrics(y_true, y_pred, model_name, test_set_name):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average="macro")
    f1_weighted = f1_score(y_true, y_pred, average="weighted")

    precision_macro, recall_macro, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    precision_weighted, recall_weighted, _, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    return {
        "model": model_name,
        "test_set": test_set_name,
        "n_eval": int(len(y_true)),
        "accuracy": float(acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }


def make_report_and_cm(y_true, y_pred):
    report_text = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=VALID_LABELS,
        digits=4,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    cm_norm_df = pd.DataFrame(
        cm_norm,
        index=[f"true_{x}" for x in VALID_LABELS],
        columns=[f"pred_{x}" for x in VALID_LABELS],
    )

    return report_text, cm_df, cm_norm_df


def print_model_result(model_name, y_true, y_pred, test_set_name="synthetic_financial_news"):
    metrics = compute_full_metrics(
        y_true=y_true,
        y_pred=y_pred,
        model_name=model_name,
        test_set_name=test_set_name
    )

    report_text, cm_df, cm_norm_df = make_report_and_cm(y_true, y_pred)

    print("\n" + "=" * 100)
    print(f"{model_name} RESULT")
    print("=" * 100)

    display(pd.DataFrame([metrics]).round(4))

    print("\nClassification Report:")
    print(report_text)

    print("\nConfusion Matrix:")
    display(cm_df)

    print("\nConfusion Matrix Normalized:")
    display(cm_norm_df.round(3))

    return metrics


In [6]:
# ------------------------------------------------------------
# 4) Original FinBERT prediction fonksiyonu
# ------------------------------------------------------------

# ProsusAI/finbert label sırası çoğunlukla:
# 0 -> positive
# 1 -> negative
# 2 -> neutral
FINBERT_ID2LABEL = {
    0: "positive",
    1: "negative",
    2: "neutral",
}

@torch.no_grad()
def predict_original_finbert(texts, model_id=FINBERT_MODEL_ID, batch_size=32, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id)

    model.to(device)
    model.eval()

    pred_ids_all = []
    pred_labels = []
    confidences = []

    score_negative = []
    score_neutral = []
    score_positive = []

    texts = [str(x) for x in texts]

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Predicting {model_id}"):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
        pred_ids = probs.argmax(axis=1)

        for pred_id, prob_vec in zip(pred_ids, probs):
            pred_id = int(pred_id)
            pred_label = FINBERT_ID2LABEL[pred_id]

            label_scores = {
                FINBERT_ID2LABEL[int(i)]: float(prob_vec[int(i)])
                for i in range(len(prob_vec))
            }

            pred_ids_all.append(pred_id)
            pred_labels.append(pred_label)
            confidences.append(float(prob_vec[pred_id]))

            score_negative.append(label_scores.get("negative", np.nan))
            score_neutral.append(label_scores.get("neutral", np.nan))
            score_positive.append(label_scores.get("positive", np.nan))

    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame({
        "pred_id_original_model": pred_ids_all,
        "pred_label": pred_labels,
        "pred_confidence": confidences,
        "score_negative": score_negative,
        "score_neutral": score_neutral,
        "score_positive": score_positive,
    })


In [7]:
# ------------------------------------------------------------
# 5) Fine-tuned model prediction fonksiyonu
# ------------------------------------------------------------

@torch.no_grad()
def predict_with_finetuned_model(model_dir, texts, batch_size=32, max_length=128):
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    model.to(device)
    model.eval()

    pred_ids_all = []
    pred_labels = []
    confidences = []

    score_negative = []
    score_neutral = []
    score_positive = []

    texts = [str(x) for x in texts]

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Predicting {Path(model_dir).parent.name}"):
        batch_texts = texts[start:start + batch_size]

        enc = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        enc = {k: v.to(device) for k, v in enc.items()}

        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
        pred_ids = probs.argmax(axis=1)

        for pred_id, prob_vec in zip(pred_ids, probs):
            pred_id = int(pred_id)

            pred_ids_all.append(pred_id)
            pred_labels.append(ID2LABEL[pred_id])
            confidences.append(float(prob_vec[pred_id]))

            score_negative.append(float(prob_vec[LABEL2ID["negative"]]))
            score_neutral.append(float(prob_vec[LABEL2ID["neutral"]]))
            score_positive.append(float(prob_vec[LABEL2ID["positive"]]))

    del model
    del tokenizer
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame({
        "pred_id": pred_ids_all,
        "pred_label": pred_labels,
        "pred_confidence": confidences,
        "score_negative": score_negative,
        "score_neutral": score_neutral,
        "score_positive": score_positive,
    })


In [8]:
# ------------------------------------------------------------
# 6) Prediction sonrası detaylı analiz fonksiyonu
# ------------------------------------------------------------

def print_prediction_details(model_name, model_eval_df):
    model_eval_df = model_eval_df.copy()

    model_eval_df["is_correct"] = model_eval_df["gold_label"] == model_eval_df["pred_label"]

    print("\nPrediction distribution:")
    print(model_eval_df["pred_label"].value_counts().reindex(VALID_LABELS))

    print("\nCorrect / Wrong:")
    print(model_eval_df["is_correct"].value_counts())
    print(model_eval_df["is_correct"].value_counts(normalize=True).mul(100).round(2))

    print("\nAccuracy by true label:")
    display(
        model_eval_df.groupby("gold_label")["is_correct"]
        .agg(["count", "mean"])
        .rename(columns={"mean": "accuracy_by_label"})
        .reindex(VALID_LABELS)
    )

    if "pred_confidence" in model_eval_df.columns:
        model_eval_df["pred_confidence"] = pd.to_numeric(
            model_eval_df["pred_confidence"],
            errors="coerce"
        )

        print("\nConfidence summary:")
        display(model_eval_df["pred_confidence"].describe())

        print("\nConfidence by correct/wrong:")
        display(
            model_eval_df.groupby("is_correct")["pred_confidence"]
            .describe()
        )

        print("\nConfidence by true label:")
        display(
            model_eval_df.groupby("gold_label")["pred_confidence"]
            .describe()
            .reindex(VALID_LABELS)
        )

    wrong_df = model_eval_df[model_eval_df["is_correct"] == False].copy()

    if "pred_confidence" in wrong_df.columns:
        wrong_df = wrong_df.sort_values("pred_confidence", ascending=False)

    print("\nWrong predictions:", wrong_df.shape)

    show_cols = [
        "annotation_id",
        "sample_id",
        "date",
        "source_file",
        "eval_text",
        "gold_label",
        "pred_label",
        "pred_confidence",
        "score_negative",
        "score_neutral",
        "score_positive",
        "text_tr",
        "reason_tr",
    ]

    show_cols = [c for c in show_cols if c in wrong_df.columns]

    print("\nHigh-confidence wrong examples:")
    display(wrong_df[show_cols].head(PREVIEW_ROWS))

    return model_eval_df


In [9]:
# ------------------------------------------------------------
# 7) Original FinBERT baseline test et
# ------------------------------------------------------------

all_metrics = []
all_prediction_dfs = {}

print("\n" + "#" * 120)
print("MODEL TEST EDİLİYOR: original_finbert")
print("Model id:", FINBERT_MODEL_ID)
print("Not: Fine-tune edilmiş model değildir; Hugging Face üzerinden doğrudan indiriliyor.")
print("#" * 120)

finbert_pred_df = predict_original_finbert(
    texts=df_eval["eval_text"].tolist(),
    model_id=FINBERT_MODEL_ID,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

finbert_eval_df = pd.concat(
    [
        df_eval.reset_index(drop=True),
        finbert_pred_df.reset_index(drop=True)
    ],
    axis=1
)

y_true_finbert = finbert_eval_df["gold_id"].astype(int).values
y_pred_finbert = finbert_eval_df["pred_label"].map(LABEL2ID).astype(int).values

metrics = print_model_result(
    model_name="original_finbert",
    y_true=y_true_finbert,
    y_pred=y_pred_finbert,
    test_set_name="synthetic_financial_news"
)

all_metrics.append(metrics)

finbert_eval_df = print_prediction_details(
    model_name="original_finbert",
    model_eval_df=finbert_eval_df
)

all_prediction_dfs["original_finbert"] = finbert_eval_df



########################################################################################################################
MODEL TEST EDİLİYOR: original_finbert
Model id: ProsusAI/finbert
Not: Fine-tune edilmiş model değildir; Hugging Face üzerinden doğrudan indiriliyor.
########################################################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting ProsusAI/finbert:   0%|          | 0/94 [00:00<?, ?it/s]


original_finbert RESULT


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,original_finbert,synthetic_financial_news,3000,0.955,0.9568,0.955,0.9551,0.9568,0.955,0.9551


...

,count,mean,std,min,25%,50%,75%,max
is_correct,,,,,,,,
False,135.0,0.762005,0.150537,0.463184,0.625342,0.803729,0.893239,0.970737
True,2865.0,0.930463,0.072926,0.437241,0.930052,0.955323,0.965770,0.975978



Confidence by true label:


,count,mean,std,min,25%,50%,75%,max
gold_label,,,,,,,,
negative,1000.0,0.955095,0.054588,0.493229,0.964200,0.970703,0.973021,0.975978
neutral,1000.0,0.865011,0.111304,0.437241,0.848268,0.912931,0.932751,0.954708
positive,1000.0,0.948539,0.039114,0.519310,0.952903,0.956107,0.957426,0.970737



Wrong predictions: (135, 27)

High-confidence wrong examples:


,annotation_id,sample_id,date,source_file,eval_text,gold_label,pred_label,pred_confidence,score_negative,score_neutral,score_positive,text_tr,reason_tr
2033,SYN_POS_0024,SYN_FIN_POS_0024,2024-03-14 00:00:00,positive_batch_003.xlsx,The semiconductor equipment maker announced a record order backlog after several chip manufacturers expanded capacity plans.,positive,negative,0.970737,0.970737,0.015546,0.013717,"Yarı iletken ekipman üreticisi, birçok çip üreticisinin kapasite artırma planlarını genişletmesinin ardından rekor sipariş birikimi açıkladı.",Rekor sipariş birikimi ve kapasite yatırımlarındaki artış gelecekteki gelir görünümünü destekler.
1840,SYN_NEU_ANN_0841,SYN_NEU_HEAD_000841,2026-04-21 00:00:00,neutral_batch_085.xlsx,A software vendor announced the completion of a previously disclosed administrative restructuring.,neutral,negative,0.954708,0.954708,0.030457,0.014834,Bir yazılım sağlayıcısı daha önce açıklanan idari yeniden yapılanmanın tamamlandığını duyurdu.,Önceden açıklanan idari sürecin tamamlanması yeni yön sinyali üretmez.
760,SYN_NEG_ANN_0761,SYN_NEG_HEAD_000761,2026-01-31 00:00:00,negative_batch_077.xlsx,The airline operator said nonperforming assets rose across several customer segments according to its latest update.,negative,positive,0.953988,0.022038,0.023973,0.953988,Söz konusu havayolu işletmecisi takipteki varlıkların birkaç müşteri segmentinde arttığını açıkladı şirketin son güncellemesine göre.,Sorunlu varlıkların artması varlık kalitesi açısından negatiftir.


In [10]:
# ------------------------------------------------------------
# 8) Fine-tuned modelleri sırayla test et
# ------------------------------------------------------------

for run in MODEL_RUNS:
    run_name = run["run_name"]
    model_dir = Path(run["model_dir"])

    if not model_dir.exists():
        print("\nUYARI: Model klasörü bulunamadı:", model_dir)
        continue

    print("\n" + "#" * 120)
    print(f"MODEL TEST EDİLİYOR: {run_name}")
    print("Model dir:", model_dir)
    print("#" * 120)

    pred_df = predict_with_finetuned_model(
        model_dir=model_dir,
        texts=df_eval["eval_text"].tolist(),
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    )

    y_true = df_eval["gold_id"].astype(int).values
    y_pred = pred_df["pred_id"].astype(int).values

    metrics = print_model_result(
        model_name=run_name,
        y_true=y_true,
        y_pred=y_pred,
        test_set_name="synthetic_financial_news"
    )

    all_metrics.append(metrics)

    base_cols = [
        "annotation_id",
        "sample_id",
        "date",
        "source_file",
        "eval_text",
        "gold_label",
        "gold_id",
    ]

    extra_cols = ["text_en", "text_tr", "reason_tr", "sector", "topic"]
    base_cols = [c for c in base_cols + extra_cols if c in df_eval.columns]

    model_eval_df = pd.concat(
        [
            df_eval[base_cols].reset_index(drop=True),
            pred_df.reset_index(drop=True)
        ],
        axis=1
    )

    model_eval_df = print_prediction_details(
        model_name=run_name,
        model_eval_df=model_eval_df
    )

    all_prediction_dfs[run_name] = model_eval_df



########################################################################################################################
MODEL TEST EDİLİYOR: roberta_base
Model dir: D:\serkan.kaymak\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\roberta_base\final_model
########################################################################################################################


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Predicting roberta_base:   0%|          | 0/94 [00:00<?, ?it/s]


roberta_base RESULT


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,roberta_base,synthetic_financial_news,3000,0.9957,0.9957,0.9957,0.9957,0.9957,0.9957,0.9957


...

,count,mean,std,min,25%,50%,75%,max
is_correct,,,,,,,,
False,125.0,0.848032,0.165616,0.391837,0.740833,0.926539,0.983350,0.997829
True,2875.0,0.990786,0.036699,0.371398,0.996744,0.997715,0.998016,0.998525



Confidence by true label:


,count,mean,std,min,25%,50%,75%,max
gold_label,,,,,,,,
negative,1000.0,0.974217,0.078635,0.371398,0.994425,0.997615,0.998057,0.998462
neutral,1000.0,0.993295,0.028352,0.586681,0.997030,0.997811,0.998084,0.998525
positive,1000.0,0.987003,0.050462,0.500401,0.996595,0.997605,0.997908,0.998147



Wrong predictions: (125, 19)

High-confidence wrong examples:


,annotation_id,sample_id,date,source_file,eval_text,gold_label,pred_label,pred_confidence,score_negative,score_neutral,score_positive,text_tr,reason_tr
317,NEG_ANN_0318,SYN_NEG_000318,2024-11-13,negative_batch_032.xlsx,"The streaming platform reported higher churn after recent price increases in the latest quarter, offsetting subscriber additions.",negative,positive,0.997829,0.000965,0.001206,0.997829,Yayın platformu son fiyat artışlarının ardından müşteri kaybının yükseldiğini bildirdi son çeyrekte; bu gelişme yeni abonelerin etkisini azalttı.,Müşteri kaybındaki artış gelir sürdürülebilirliği açısından negatiftir.
337,NEG_ANN_0338,SYN_NEG_000338,2024-12-03,negative_batch_034.xlsx,"The streaming platform reported higher churn after recent price increases for the current fiscal year, offsetting subscriber additions.",negative,positive,0.997820,0.000953,0.001227,0.997820,Yayın platformu son fiyat artışlarının ardından müşteri kaybının yükseldiğini bildirdi mevcut mali yıl için; bu gelişme yeni abonelerin etkisini azalttı.,Müşteri kaybındaki artış gelir sürdürülebilirliği açısından negatiftir.
357,NEG_ANN_0358,SYN_NEG_000358,2024-12-23,negative_batch_036.xlsx,"The streaming platform reported higher churn after recent price increases after management reviewed order trends, offsetting subscriber additions.",negative,positive,0.997806,0.000989,0.001206,0.997806,Yayın platformu son fiyat artışlarının ardından müşteri kaybının yükseldiğini bildirdi yönetimin sipariş eğilimlerini değerlendirmesinin ardından; bu gelişme yeni abonelerin et...,Müşteri kaybındaki artış gelir sürdürülebilirliği açısından negatiftir.


In [11]:
# ------------------------------------------------------------
# 9) Summary tablo
# ------------------------------------------------------------

summary_df = pd.DataFrame(all_metrics)

if summary_df.empty:
    print("Hiçbir model başarıyla test edilemedi.")
else:
    summary_df = summary_df.sort_values(
        "f1_macro",
        ascending=False
    ).reset_index(drop=True)

    print("\n" + "=" * 120)
    print("FINAL SUMMARY ON SYNTHETIC FINANCIAL NEWS SET")
    print("=" * 120)

    display(summary_df.round(4))

    print("\nMarkdown tablo:")
    display(summary_df.round(4))

    best = summary_df.iloc[0]

    print("\nBest model:")
    print("model:", best["model"])
    print("accuracy:", round(float(best["accuracy"]), 4))
    print("f1_macro:", round(float(best["f1_macro"]), 4))
    print("f1_weighted:", round(float(best["f1_weighted"]), 4))



FINAL SUMMARY ON SYNTHETIC FINANCIAL NEWS SET


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,roberta_base,synthetic_financial_news,3000,0.9957,0.9957,0.9957,0.9957,0.9957,0.9957,0.9957
1,bert_base_uncased,synthetic_financial_news,3000,0.9743,0.9748,0.9743,0.9742,0.9748,0.9743,0.9742
2,distilbert_base_uncased,synthetic_financial_news,3000,0.9583,0.9591,0.9583,0.9581,0.9591,0.9583,0.9581
3,original_finbert,synthetic_financial_news,3000,0.9550,0.9568,0.9550,0.9551,0.9568,0.9550,0.9551



Markdown tablo:


,model,test_set,n_eval,accuracy,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted
0,roberta_base,synthetic_financial_news,3000,0.9957,0.9957,0.9957,0.9957,0.9957,0.9957,0.9957
1,bert_base_uncased,synthetic_financial_news,3000,0.9743,0.9748,0.9743,0.9742,0.9748,0.9743,0.9742
2,distilbert_base_uncased,synthetic_financial_news,3000,0.9583,0.9591,0.9583,0.9581,0.9591,0.9583,0.9581
3,original_finbert,synthetic_financial_news,3000,0.9550,0.9568,0.9550,0.9551,0.9568,0.9550,0.9551



Best model:
model: roberta_base
accuracy: 0.9957
f1_macro: 0.9957
f1_weighted: 0.9957
